## Dataset Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics==8.4.216
!cp "/content/drive/MyDrive/ معكسر علم البيانات والذكاء الاصطماعي/sds_swimmer.zip" /content/
!unzip -q -o /content/sds_swimmer.zip -d /content/

# Rebuild the clean data.yaml
import os, glob, yaml
yml = glob.glob('/content/**/data.yaml', recursive=True)[0]
DATA_ROOT = os.path.dirname(yml)

def find_split_dir(split):
    for pattern in [f'{DATA_ROOT}/**/{split}/images', f'{DATA_ROOT}/**/images/{split}']:
        hits = glob.glob(pattern, recursive=True)
        if hits:
            return os.path.abspath(hits[0])
    return None

splits = {s: find_split_dir(s) for s in ['train', 'val', 'test']}
splits['val'] = splits['val'] or find_split_dir('valid')
print(splits)
assert all(splits.values()), "A split folder is missing."

yaml.safe_dump({'path': '/', 'train': splits['train'], 'val': splits['val'],
                'test': splits['test'], 'names': {0: 'swimmer'}},
               open('/content/swimmer_data.yaml', 'w'), sort_keys=False)

# Check the baseline weights
BASELINE_W = "/content/drive/MyDrive/ معكسر علم البيانات والذكاء الاصطماعي/runs_cv/yolo26s_baseline/weights/best.pt"
print("Baseline found:", os.path.exists(BASELINE_W))

Mounted at /content/drive
{'train': '/content/sds_swimmer/train/images', 'val': '/content/sds_swimmer/val/images', 'test': '/content/sds_swimmer/test/images'}
Baseline found: True


## Configuration

In [ ]:
import os, gc, glob, json, shutil, yaml
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix
from IPython.display import display


DRIVE_DIR   = "/content/drive/MyDrive/ معكسر علم البيانات والذكاء الاصطماعي"
PROJECT_DIR = f"{DRIVE_DIR}/runs_cv"
MODEL_NAME  = 'yolo26s'
RUN_NAME    = f'{MODEL_NAME}_baseline'
DATA_YAML   = '/content/swimmer_data.yaml'
CONF        = 0.25

BASELINE_W = f'{PROJECT_DIR}/{RUN_NAME}/weights/best.pt' # 30-epoch baseline
assert os.path.exists(BASELINE_W), "Baseline best.pt not found"
assert os.path.exists(DATA_YAML), "Run the data.yaml"

EXTRA_EPOCHS = [5, 10]
IMG_SIZES    = [640, 800]


TUNE_EXTRA = {}

grid = [(e, s) for e in EXTRA_EPOCHS for s in IMG_SIZES]
print(grid)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
[(5, 640), (5, 800), (10, 640), (10, 800)]


### Fine-Tuning

In [ ]:
def epochs_done(run_dir):
    csv = f'{run_dir}/results.csv'
    return len(pd.read_csv(csv)) if os.path.exists(csv) else 0

tune_runs = {}
for extra_ep, sz in grid:
    name = f'{MODEL_NAME}_ft_e{extra_ep}_{sz}'
    run_dir = f'{PROJECT_DIR}/{name}'
    tune_runs[name] = dict(extra_epochs=extra_ep, imgsz=sz, run_dir=run_dir)

    done = epochs_done(run_dir)
    if done >= extra_ep:
        print(f'✓ {name} finished ({done}/{extra_ep} epochs) — skipping')
        continue

    if done > 0 and os.path.exists(f'{run_dir}/weights/last.pt'):
        print(f'\n↻ {name}: resuming from epoch {done}/{extra_ep}')
        YOLO(f'{run_dir}/weights/last.pt').train(resume=True)
    else:
        print(f'\n===== {name}: +{extra_ep} epochs @ {sz} =====')
        YOLO(BASELINE_W).train(data=DATA_YAML, epochs=extra_ep, imgsz=sz,
                               project=PROJECT_DIR, name=name, exist_ok=True, **TUNE_EXTRA)
    gc.collect(); torch.cuda.empty_cache()


===== yolo26s_ft_e5_640: +5 epochs @ 640 =====
New https://pypi.org/project/ultralytics/8.4.218 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/swimmer_data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup

### Evaluation

In [ ]:
def eval_run(weights, imgsz, label, extra_ep):
    m_ = YOLO(weights)
    rows = []
    for split in ['val', 'test']:
        m = m_.val(data=DATA_YAML, split=split, imgsz=imgsz, batch=8, plots=(split == 'test'),
                   project=PROJECT_DIR, name=f'{label}_eval_{split}', exist_ok=True, verbose=False)
        p, r = m.box.mp, m.box.mr
        rows.append({'run': label, 'total_epochs': 30 + extra_ep, 'imgsz': imgsz, 'split': split,
                     'precision': p, 'recall': r, 'f1': 2*p*r/(p+r+1e-9),
                     'mAP50': m.box.map50, 'mAP50-95': m.box.map,
                     'inference_ms_per_img': m.speed['inference']})
    del m_; gc.collect(); torch.cuda.empty_cache()
    return rows

all_rows = eval_run(BASELINE_W, 640, f'{MODEL_NAME}_baseline', 0)
for name, info in tune_runs.items():
    all_rows += eval_run(f"{info['run_dir']}/weights/best.pt", info['imgsz'], name, info['extra_epochs'])

tune_df = pd.DataFrame(all_rows)
tune_df.insert(0, 'model', MODEL_NAME)
tune_df.to_csv(f'{PROJECT_DIR}/{MODEL_NAME}_tuning_metrics.csv', index=False)

cols = ['run', 'total_epochs', 'imgsz', 'precision', 'recall', 'f1', 'mAP50', 'mAP50-95', 'inference_ms_per_img']
print('VALIDATION'); display(tune_df[tune_df.split == 'val'][cols].round(4).reset_index(drop=True))
print('TEST');       display(tune_df[tune_df.split == 'test'][cols].round(4).reset_index(drop=True))

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s summary (fused): 120 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2585.8±574.5 MB/s, size: 456.6 KB)
val: Scanning /content/sds_swimmer/val/labels.cache... 1000 images, 160 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1000/1000 322.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 125/125 3.1it/s 40.6s
                   all       1000       4068      0.852      0.819      0.811      0.344
Speed: 1.2ms preprocess, 7.2ms inference, 0.0ms loss, 2.4ms postprocess per image
Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s summary (fused): 120 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 120.3±48.7 MB/s, size: 679.6 KB)
val: Scanning /content/sds_swimmer/test/labels... 1000 

,run,total_epochs,imgsz,precision,recall,f1,mAP50,mAP50-95,inference_ms_per_img
0,yolo26s_baseline,30,640,0.8517,0.8185,0.8348,0.8111,0.3435,7.2425
1,yolo26s_ft_e5_640,35,640,0.8420,0.7908,0.8156,0.7950,0.3333,7.0620
2,yolo26s_ft_e5_800,35,800,0.8625,0.8328,0.8474,0.8238,0.3587,9.2906
3,yolo26s_ft_e10_640,40,640,0.8421,0.8090,0.8252,0.7971,0.3341,6.3986
4,yolo26s_ft_e10_800,40,800,0.8592,0.8309,0.8448,0.8219,0.3600,9.1009


TEST


,run,total_epochs,imgsz,precision,recall,f1,mAP50,mAP50-95,inference_ms_per_img
0,yolo26s_baseline,30,640,0.8330,0.8089,0.8208,0.8002,0.3401,6.7830
1,yolo26s_ft_e5_640,35,640,0.8234,0.7889,0.8058,0.7811,0.3255,6.9689
2,yolo26s_ft_e5_800,35,800,0.8368,0.8160,0.8263,0.8023,0.3439,9.2791
3,yolo26s_ft_e10_640,40,640,0.8210,0.7964,0.8085,0.7800,0.3276,6.9919
4,yolo26s_ft_e10_800,40,800,0.8459,0.8286,0.8371,0.8092,0.3537,9.2475
